# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [63]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
spaceship.dropna(inplace=True)

In [4]:
spaceship.isna().sum()

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64

In [5]:
spaceship.shape

(6606, 14)

Now perform the same as before:
- Feature Scaling
- Feature Selection


- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [6]:
spaceship['Cabin'] = spaceship['Cabin'].str.split('/').str[0]
spaceship

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False


- Drop PassengerId and Name

In [7]:
spaceship = spaceship.drop(columns=['PassengerId', 'Name'])

In [8]:
spaceship[['CryoSleep', 'VIP', 'Transported']] = spaceship[['CryoSleep', 'VIP', 'Transported']].astype(int)

In [9]:
spaceship

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,0,B,TRAPPIST-1e,39.0,0,0.0,0.0,0.0,0.0,0.0,0
1,Earth,0,F,TRAPPIST-1e,24.0,0,109.0,9.0,25.0,549.0,44.0,1
2,Europa,0,A,TRAPPIST-1e,58.0,1,43.0,3576.0,0.0,6715.0,49.0,0
3,Europa,0,A,TRAPPIST-1e,33.0,0,0.0,1283.0,371.0,3329.0,193.0,0
4,Earth,0,F,TRAPPIST-1e,16.0,0,303.0,70.0,151.0,565.0,2.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
8688,Europa,0,A,55 Cancri e,41.0,1,0.0,6819.0,0.0,1643.0,74.0,0
8689,Earth,1,G,PSO J318.5-22,18.0,0,0.0,0.0,0.0,0.0,0.0,0
8690,Earth,0,G,TRAPPIST-1e,26.0,0,0.0,0.0,1872.0,1.0,0.0,1
8691,Europa,0,E,55 Cancri e,32.0,0,0.0,1049.0,0.0,353.0,3235.0,0


**Perform Train Test Split**

In [10]:
features = spaceship.drop(columns = ["Transported"])
target = spaceship["Transported"]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.2, random_state = 17)

- Encoding Categorical Columns

In [12]:
X_train.select_dtypes("object")

C:\Users\Admin\AppData\Local\Temp\ipykernel_11556\3378186967.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X_train.select_dtypes("object")


,HomePlanet,Cabin,Destination
1388,Europa,B,55 Cancri e
1031,Mars,E,TRAPPIST-1e
3717,Earth,G,TRAPPIST-1e
2829,Earth,F,TRAPPIST-1e
6236,Earth,G,TRAPPIST-1e
...,...,...,...
1792,Earth,G,TRAPPIST-1e
536,Europa,C,TRAPPIST-1e
7279,Mars,F,TRAPPIST-1e
2934,Earth,F,TRAPPIST-1e


In [13]:
spaceship["Destination"].value_counts()

Destination
TRAPPIST-1e      4576
55 Cancri e      1407
PSO J318.5-22     623
Name: count, dtype: int64

In [14]:
spaceship["HomePlanet"].value_counts()

HomePlanet
Earth     3566
Europa    1673
Mars      1367
Name: count, dtype: int64

In [30]:
# Encoding HomePlanet
encoder_hp = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded_array = encoder_hp.fit_transform(X_train[["HomePlanet"]])
encoded_columns = encoder_hp.get_feature_names_out(["HomePlanet"])
encoded_df = pd.DataFrame(encoded_array, columns=encoded_columns, index=X_train.index)
X_train_encoded = pd.concat([X_train, encoded_df], axis=1)

In [31]:
test_encoded_array = encoder_hp.transform(X_test[["HomePlanet"]])
test_encoded_columns = encoder_hp.get_feature_names_out(["HomePlanet"])
test_encoded_df = pd.DataFrame(test_encoded_array, columns=test_encoded_columns, index=X_test.index)
X_test_encoded = pd.concat([X_test, test_encoded_df], axis=1)

In [32]:
# Encoding HomePlanet
encoder_dest = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded_array = encoder_dest.fit_transform(X_train[["Destination"]])
encoded_columns = encoder_dest.get_feature_names_out(["Destination"])
encoded_df = pd.DataFrame(encoded_array, columns=encoded_columns, index=X_train.index)
X_train_encoded = pd.concat([X_train_encoded, encoded_df], axis=1)

In [33]:
test_encoded_array = encoder_dest.transform(X_test[["Destination"]])
test_encoded_columns = encoder_dest.get_feature_names_out(["Destination"])
test_encoded_df = pd.DataFrame(test_encoded_array, columns=test_encoded_columns, index=X_test.index)
X_test_encoded = pd.concat([X_test_encoded, test_encoded_df], axis=1)

In [34]:
# Encoding HomePlanet
encoder_cabin = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded_array = encoder_cabin.fit_transform(X_train[["Cabin"]])
encoded_columns = encoder_cabin.get_feature_names_out(["Cabin"])
encoded_df = pd.DataFrame(encoded_array, columns=encoded_columns, index=X_train.index)
X_train_encoded = pd.concat([X_train_encoded, encoded_df], axis=1)

In [35]:
test_encoded_array = encoder_cabin.transform(X_test[["Cabin"]])
test_encoded_columns = encoder_cabin.get_feature_names_out(["Cabin"])
test_encoded_df = pd.DataFrame(test_encoded_array, columns=test_encoded_columns, index=X_test.index)
X_test_encoded = pd.concat([X_test_encoded, test_encoded_df], axis=1)

In [41]:
X_train_encoded.columns

Index(['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP',
       'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'HomePlanet_Earth', 'HomePlanet_Europa', 'HomePlanet_Mars',
       'Destination_55 Cancri e', 'Destination_PSO J318.5-22',
       'Destination_TRAPPIST-1e', 'Cabin_A', 'Cabin_B', 'Cabin_C', 'Cabin_D',
       'Cabin_E', 'Cabin_F', 'Cabin_G', 'Cabin_T'],
      dtype='str')

In [38]:
X_test_encoded.columns

Index(['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP',
       'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'HomePlanet_Earth', 'HomePlanet_Europa', 'HomePlanet_Mars',
       'Destination_55 Cancri e', 'Destination_PSO J318.5-22',
       'Destination_TRAPPIST-1e', 'Cabin_A', 'Cabin_B', 'Cabin_C', 'Cabin_D',
       'Cabin_E', 'Cabin_F', 'Cabin_G', 'Cabin_T'],
      dtype='str')

In [44]:
X_train_encoded = X_train_encoded.drop(columns=['HomePlanet', 'Destination', 'Cabin'])

In [45]:
X_test_encoded = X_test_encoded.drop(columns=['HomePlanet', 'Destination', 'Cabin'])

In [47]:
#checking if all values are numerical now
X_train_encoded.info()

<class 'pandas.DataFrame'>
Index: 5284 entries, 1388 to 3558
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CryoSleep                  5284 non-null   int64  
 1   Age                        5284 non-null   float64
 2   VIP                        5284 non-null   int64  
 3   RoomService                5284 non-null   float64
 4   FoodCourt                  5284 non-null   float64
 5   ShoppingMall               5284 non-null   float64
 6   Spa                        5284 non-null   float64
 7   VRDeck                     5284 non-null   float64
 8   HomePlanet_Earth           5284 non-null   float64
 9   HomePlanet_Europa          5284 non-null   float64
 10  HomePlanet_Mars            5284 non-null   float64
 11  Destination_55 Cancri e    5284 non-null   float64
 12  Destination_PSO J318.5-22  5284 non-null   float64
 13  Destination_TRAPPIST-1e    5284 non-null   float64
 14  Cabin

- Normalization

In [53]:
normalizer = MinMaxScaler()

In [54]:
normalizer.fit(X_train_encoded)

,"feature_range feature_range: tuple (min, max), default=(0, 1)Desired range of transformed data.","(0, ...)"
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array).",True
,"clip clip: bool, default=FalseSet to True to clip transformed values of held-out data toprovided `feature_range`.Since this parameter will clip values, `inverse_transform` may notbe able to restore the original data... note:: Setting `clip=True` does not prevent feature drift (a distribution shift between training and test data). The transformed values are clipped to the `feature_range`, which helps avoid unintended behavior in models sensitive to out-of-range inputs (e.g. linear models). Use with care, as clipping can distort the distribution of test data... versionadded:: 0.24",False
Name,Type,Value
"data_max_ data_max_: ndarray of shape (n_features,)Per feature maximum seen in the data.. versionadded:: 0.17 *data_max_*","ndarray[float64](22,)","[ 1.,79., 1.,..., 1., 1., 1.]"
"data_min_ data_min_: ndarray of shape (n_features,)Per feature minimum seen in the data.. versionadded:: 0.17 *data_min_*","ndarray[float64](22,)","[0.,0.,0.,...,0.,0.,0.]"
"data_range_ data_range_: ndarray of shape (n_features,)Per feature range ``(data_max_ - data_min_)`` seen in the data.. versionadded:: 0.17 *data_range_*","ndarray[float64](22,)","[ 1.,79., 1.,..., 1., 1., 1.]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](22,)","['CryoSleep','Age','VIP',...,'Cabin_F','Cabin_G','Cabin_T']"
"min_ min_: ndarray of shape (n_features,)Per feature adjustment for minimum. Equivalent to``min - X.min(axis=0) * self.scale_``","ndarray[float64](22,)","[0.,0.,0.,...,0.,0.,0.]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,22
"n_samples_seen_ n_samples_seen_: intThe number of samples processed by the estimator.It will be reset on new calls to fit, but increments across``partial_fit`` calls.",int,5284


In [55]:
X_train_norm = normalizer.transform(X_train_encoded)
X_test_norm = normalizer.transform(X_test_encoded)

In [56]:
X_train_norm = pd.DataFrame(X_train_norm, columns = X_train_encoded.columns)
X_test_norm = pd.DataFrame(X_test_norm, columns = X_test_encoded.columns)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- KNN Classifier

In [67]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_norm, y_train)
pred = model.predict(X_test_norm)
accuracy_score(y_test, pred)

0.773071104387292

- Bagging and Pasting

In [64]:
model = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, max_samples = 1000)
model.fit(X_train_norm, y_train)
pred = model.predict(X_test_norm)
accuracy_score(pred, y_test)

0.8040847201210287

- Random Forests

In [65]:
model = RandomForestClassifier(n_estimators=100, max_depth=20)
model.fit(X_train_norm, y_train)
pred = model.predict(X_test_norm)
accuracy_score(pred, y_test)

0.8055975794251135

- Gradient Boosting

In [70]:
model = GradientBoostingClassifier(max_depth=5, n_estimators=100)
model.fit(X_train_norm, y_train)
pred = model.predict(X_test_norm)
accuracy_score(pred, y_test)

0.8184568835098336

- Adaptive Boosting

In [71]:
model = AdaBoostClassifier(DecisionTreeClassifier(max_depth=5),n_estimators=100)
model.fit(X_train_norm, y_train)
pred = model.predict(X_test_norm)
accuracy_score(pred, y_test)

0.789712556732224

Which model is the best and why?

Gradient Boosting Classifier achieved the highest test accuracy (~81.85%), outperforming Random Forest (~80.56%), Bagging (~80.41%), AdaBoost (~78.97%), and KNN (~77.31%).

Why:
- Sequential learning: Gradient Boosting constructs trees additively, directly optimizing the loss function and correcting errors made by previous estimators.
- Strong tabular performance: It effectively captures subtle non-linear interactions across mixed tabular features better than simple distance-based models (KNN) or parallel ensemble averages (Random Forest/Bagging).